# TP2 — Named Entity Recognition with an LSTM

Adapted from `SentimentModelLSTM` in `LSTM_classification.py`, which does **sentence**
classification (sentiment). NER is **token** classification, so the same four changes as in
the CNN notebook apply:

| | `LSTM_classification.py` | this notebook |
|---|---|---|
| Data | CSV, one `review` + one `label` per row | CoNLL, one token + one tag per line |
| Output | `o[:, -1, :]` — **last** time step only | every time step, one tag per token |
| Loss | `sigmoid` + `BCELoss`, 2 classes | `cross_entropy` on 21 tags, ignoring padding |
| Metric | `sklearn.classification_report` | `seqeval` (entity-level P / R / F1) |

Everything outside the model class — data loading, vocabulary, embedding matrices, training
loop, evaluation — is identical to `cnn.ipynb`, so the two sets of scores are comparable.

Two LSTM-specific points, both explained where they occur:

- the LSTM is **bidirectional**, otherwise each tag would be predicted from the left context only;
- the padding is removed with `pack_padded_sequence` before the LSTM runs.


## 0. Requirements


In [17]:
# on Google Colab, run this first
# %pip install torch pandas scikit-learn gensim seqeval tqdm


In [18]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 1. Configuration


In [19]:
DATA = Path('../TP_ISD2020/QUAERO_FrenchMed')
EMBEDDINGS_DIR = Path('../models')          # the models trained in TP1

CORPUS = 'EMEA'                         # 'EMEA' (drug leaflets) or 'MEDLINE' (article titles)

train_file = DATA / CORPUS / f'{CORPUS}train_layer1_ID.conll'
valid_file = DATA / CORPUS / f'{CORPUS}dev_layer1_ID.conll'
test_file  = DATA / CORPUS / f'{CORPUS}test_layer1_ID.conll'

sequence_length = 128    # longest sentence in the corpus is 121 tokens, so nothing is truncated
max_vocab = 32000
embedding_size = 100     # must match the TP1 embeddings (dim = 100)

# LSTM hyper-parameters (the defaults of the original script)
hidden_size = 128
n_layers = 2
bidirectional = True     # see section 4

batch_size = 32
epochs = 30
lr = 1e-3
dropout = 0.25
grad_clip = 5
es_limit = 5             # early stopping patience

IGNORE_INDEX = -100      # tag id given to padding, ignored by the loss

## 2. Loading the CoNLL data

Each line has 5 whitespace-separated columns: token index, token, two unused columns,
and the NER tag. An empty line separates two sentences.


In [20]:
def read_conll(path):
    """Read a CoNLL file as (sentences, tags), both lists of lists of strings."""
    sentences, tags = [], []
    tokens, labels = [], []

    for line in path.read_text(encoding='utf-8').splitlines():
        columns = line.split()
        if not columns:                  # empty line -> end of a sentence
            if tokens:
                sentences.append(tokens)
                tags.append(labels)
                tokens, labels = [], []
            continue
        tokens.append(columns[1])        # column 2 = the token
        labels.append(columns[4])        # column 5 = the NER tag

    if tokens:                           # last sentence, if the file has no trailing blank line
        sentences.append(tokens)
        tags.append(labels)

    return sentences, tags


train_sents, train_tags = read_conll(train_file)
valid_sents, valid_tags = read_conll(valid_file)
test_sents,  test_tags  = read_conll(test_file)

print(f'{CORPUS}: {len(train_sents)} train / {len(valid_sents)} dev / {len(test_sents)} test sentences')
print()
for token, tag in list(zip(train_sents[2], train_tags[2])):
    print(f'  {token:12s} {tag}')


EMEA: 706 train / 649 dev / 578 test sentences

  Qu           O
  ’            O
  est          O
  ce           O
  que          O
  Prialt       B-CHEM
  ?            O


### The tag set


In [21]:
# the tag vocabulary is built on the training set (dev and test use no other tag)
TAGS = sorted({t for tags in train_tags for t in tags})
tag2int = {t: i for i, t in enumerate(TAGS)}
int2tag = {i: t for t, i in tag2int.items()}
num_classes = len(TAGS)

print(f'{num_classes} tags:', ' '.join(TAGS))


21 tags: B-ANAT B-CHEM B-DEVI B-DISO B-GEOG B-LIVB B-OBJC B-PHEN B-PHYS B-PROC I-ANAT I-CHEM I-DEVI I-DISO I-GEOG I-LIVB I-OBJC I-PHEN I-PHYS I-PROC O


### Vocabulary and encoding

Tag padding gets `IGNORE_INDEX` so those positions are skipped by the loss and by the metrics.


In [22]:
def build_vocab(*sentence_lists, max_vocab=-1):
    """Build word2int from all splits, as the original script does."""
    words = [w for sents in sentence_lists for s in sents for w in s]
    counter = Counter(words)
    vocab = sorted(counter, key=counter.get, reverse=True)
    if max_vocab != -1:
        vocab = vocab[:max_vocab]
    int2word = dict(enumerate(vocab, 2))
    int2word[0] = '<PAD>'
    int2word[1] = '<UNK>'
    return {word: i for i, word in int2word.items()}


def encode(sentences, tags, word2int, seq_length=128):
    """Turn sentences and tags into two padded (n_sentences, seq_length) int arrays."""
    x = np.zeros((len(sentences), seq_length), dtype=np.int64)              # 0 = <PAD>
    y = np.full((len(sentences), seq_length), IGNORE_INDEX, dtype=np.int64)

    for i, (sentence, labels) in enumerate(zip(sentences, tags)):
        ids = [word2int.get(w, 1) for w in sentence][:seq_length]           # 1 = <UNK>
        lab = [tag2int[t] for t in labels][:seq_length]
        x[i, :len(ids)] = ids
        y[i, :len(lab)] = lab

    return x, y


vocab_index = build_vocab(train_sents, valid_sents, test_sents, max_vocab=max_vocab)
vocab_size = len(vocab_index)

train_x, train_y = encode(train_sents, train_tags, vocab_index, sequence_length)
valid_x, valid_y = encode(valid_sents, valid_tags, vocab_index, sequence_length)
test_x,  test_y  = encode(test_sents,  test_tags,  vocab_index, sequence_length)

print('Taille vocabulaire', vocab_size)
print('Feature Shapes:')
print('  Train set:     ', train_x.shape)
print('  Validation set:', valid_x.shape)
print('  Test set:      ', test_x.shape)


Taille vocabulaire 4590
Feature Shapes:
  Train set:      (706, 128)
  Validation set: (649, 128)
  Test set:       (578, 128)


In [23]:
trainset = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
validset = TensorDataset(torch.from_numpy(valid_x), torch.from_numpy(valid_y))
testset  = TensorDataset(torch.from_numpy(test_x),  torch.from_numpy(test_y))

trainloader = DataLoader(trainset, shuffle=True,  batch_size=batch_size)
valloader   = DataLoader(validset, shuffle=False, batch_size=batch_size)
testloader  = DataLoader(testset,  shuffle=False, batch_size=batch_size)


## 3. Initialising the embeddings with the TP1 models

In the original script the embedding layer is `nn.Embedding(V, D)`, initialised randomly.
Here we build an embedding **matrix** from a TP1 model and copy it into that layer.

For fastText we load the `.bin`, so a word missing from the embedding vocabulary still gets a
vector built from its character n-grams. word2vec has no such fallback, and those words keep
their random initialisation.

In [24]:
from gensim.models import Word2Vec
from gensim.models.fasttext import load_facebook_vectors

# the 6 models trained in TP1, plus a random baseline
EMBEDDINGS = {
    'random':             None,
    'med_w2v_cbow':       EMBEDDINGS_DIR / 'med_w2v_cbow.model',
    'med_w2v_skipgram':   EMBEDDINGS_DIR / 'med_w2v_skipgram.model',
    'med_ft_cbow':        EMBEDDINGS_DIR / 'med_ft_cbow.bin',
    'press_w2v_cbow':     EMBEDDINGS_DIR / 'press_w2v_cbow.model',
    'press_w2v_skipgram': EMBEDDINGS_DIR / 'press_w2v_skipgram.model',
    'press_ft_cbow':      EMBEDDINGS_DIR / 'press_ft_cbow.bin',
}


def load_vectors(path):
    """Load a TP1 model as gensim KeyedVectors."""
    if path.suffix == '.bin':
        return load_facebook_vectors(str(path))   # fastText, can vectorise unknown words
    return Word2Vec.load(str(path)).wv            # word2vec


def build_embedding_matrix(path, word2int, dim=embedding_size):
    """Build a (vocab_size, dim) matrix, one row per word of our NER vocabulary."""
    kv = load_vectors(path)

    # words not covered keep a small random vector; <PAD> (row 0) stays at zero
    matrix = np.random.normal(0, 0.1, (len(word2int), dim)).astype(np.float32)
    matrix[0] = 0

    found = 0
    for word, i in word2int.items():
        if i < 2:                 # skip <PAD> and <UNK>
            continue
        try:
            matrix[i] = kv[word]  # for fastText this also works for unknown words
            found += 1
        except KeyError:
            pass

    coverage = found / (len(word2int) - 2)
    return matrix, coverage

## 4. The model

Three changes to `SentimentModelLSTM`.

**1. Every time step, not just the last.** The original keeps `o[:, -1, :]`, one vector for the
whole sentence. For NER the full `(N, W, H)` output is kept and the linear layer is applied to
each position.

**2. No sigmoid.** `cross_entropy` expects raw logits over the 21 tags, so the final
`sigmoid` of the binary version is removed and `fc` outputs `num_classes`.

**3. Bidirectional, over unpadded sequences.** A left-to-right LSTM would tag each token from
its left context only, while the CNN sees both sides; `bidirectional=True` makes the two
comparable, and `fc` then takes `2 * hidden_size` inputs. This makes the padding matter: the
backward pass starts at the end of the row, so it would run through ~110 padding steps before
reaching the real tokens. `pack_padded_sequence` hides the padding from the LSTM, and
`pad_packed_sequence` restores the fixed width afterwards.


In [25]:
class NERModelLSTM(nn.Module):
    def __init__(self, vocab_size, class_size, hidden_size=128, embedding_size=100,
                 n_layers=2, dropout=0.2, bidirectional=True,
                 pretrained=None, freeze=False):
        super(NERModelLSTM, self).__init__()
        self.name = 'lstm'

        # embedding layer is useful to map input into vector representation
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=0)
        if pretrained is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained))
            self.embedding.weight.requires_grad = not freeze

        # LSTM layer preserved by PyTorch library (dropout only applies between layers)
        self.lstm = nn.LSTM(embedding_size, hidden_size, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0,
                            bidirectional=bidirectional)

        # dropout layer
        self.dropout = nn.Dropout(0.3)

        # Linear layer for output: one score per tag, for each token
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), class_size)

    def forward(self, x):
        x = x.long()
        width = x.size(1)

        # number of real tokens per sentence (id 0 is <PAD>)
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()

        # map input to vector
        e = self.embedding(x)

        # pass forward to lstm, without the padding
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths, batch_first=True,
                                                   enforce_sorted=False)
        o, _ = self.lstm(packed)
        o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=width)

        # apply dropout and fully connected layer, keeping every time step
        o = self.dropout(o)                # (N, W, H * n_directions)
        o = self.fc(o)                     # (N, W, C)  -> one tag per token

        return o


## 5. Training and evaluation

Identical to the CNN notebook. The loss flattens the logits to `(N*W, C)` and the targets to
`(N*W,)`, and `ignore_index` drops the padding positions.


In [26]:
def compute_loss(logits, targets):
    """Cross-entropy over every real token of the batch, padding excluded."""
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),   # (N*W, C)
        targets.reshape(-1),                   # (N*W,)
        ignore_index=IGNORE_INDEX,
    )


def token_accuracy(logits, targets):
    """Share of correctly tagged tokens, padding excluded."""
    predicted = logits.argmax(dim=-1)
    mask = targets != IGNORE_INDEX
    return (predicted[mask] == targets[mask]).float().mean().item()


def predict_tags(model, loader):
    """Run the model and return (true_tags, predicted_tags) as lists of tag sequences."""
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for feature, target in loader:
            feature, target = feature.to(device), target.to(device)
            predicted = model(feature).argmax(dim=-1)

            for gold_row, pred_row in zip(target.cpu().numpy(), predicted.cpu().numpy()):
                keep = gold_row != IGNORE_INDEX          # drop the padding
                y_true.append([int2tag[i] for i in gold_row[keep]])
                y_pred.append([int2tag[i] for i in pred_row[keep]])

    return y_true, y_pred


In [27]:
def train_model(pretrained=None, freeze=False, epochs=epochs, verbose=True):
    """Train one LSTM and return it, together with its training history."""
    model = NERModelLSTM(vocab_size, num_classes, hidden_size, embedding_size,
                         n_layers, dropout, bidirectional,
                         pretrained=pretrained, freeze=freeze).to(device)
    optim = Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    val_loss_min = torch.inf
    best_state = None
    es_trigger = 0

    epochloop = tqdm(range(epochs), desc='Training', leave=False)
    for e in epochloop:

        # training
        model.train()
        train_loss = train_acc = 0
        for feature, target in trainloader:
            feature, target = feature.to(device), target.to(device)

            optim.zero_grad()
            out = model(feature)
            loss = compute_loss(out, target)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optim.step()

            train_loss += loss.item()
            train_acc += token_accuracy(out, target)

        # validation
        model.eval()
        val_loss = val_acc = 0
        with torch.no_grad():
            for feature, target in valloader:
                feature, target = feature.to(device), target.to(device)
                out = model(feature)
                val_loss += compute_loss(out, target).item()
                val_acc += token_accuracy(out, target)

        history['train_loss'].append(train_loss / len(trainloader))
        history['train_acc'].append(train_acc / len(trainloader))
        history['val_loss'].append(val_loss / len(valloader))
        history['val_acc'].append(val_acc / len(valloader))
        epochloop.set_postfix_str(f"val loss {history['val_loss'][-1]:.3f}")

        # keep the best model, and stop if the validation loss stops improving
        if history['val_loss'][-1] <= val_loss_min:
            val_loss_min = history['val_loss'][-1]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            es_trigger = 0
        else:
            es_trigger += 1
            if es_trigger >= es_limit:
                if verbose:
                    print(f'Early stopped at epoch {e + 1}')
                break

    model.load_state_dict(best_state)
    return model, history


### A first run, with random embeddings

This is the baseline: the model of the original script, with its randomly initialised
embedding layer.


In [28]:
model, history = train_model(pretrained=None)
print(model)

y_true, y_pred = predict_tags(model, testloader)
print(classification_report(y_true, y_pred, digits=4, zero_division=0))


Early stopped at epoch 14
NERModelLSTM(
  (embedding): Embedding(4590, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.25, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=21, bias=True)
)
              precision    recall  f1-score   support

        ANAT     0.0000    0.0000    0.0000       109
        CHEM     0.2381    0.1509    0.1847       762
        DEVI     0.0000    0.0000    0.0000        76
        DISO     0.1772    0.1807    0.1789       249
        GEOG     0.0000    0.0000    0.0000        14
        LIVB     0.8012    0.5536    0.6548       233
        OBJC     0.0000    0.0000    0.0000        49
        PHEN     0.0000    0.0000    0.0000        29
        PHYS     0.0000    0.0000    0.0000        73
        PROC     0.6013    0.4035    0.4829       228

   micro avg     0.3625    0.2091    0.2652      1822
   macro avg     0.1818    0.1289    0.1501      1822
weighted 

## 6. One model per embeddings type

Each of the 6 TP1 models is used twice, to separate two things that the earlier runs mixed up:

- **fine-tuned** (`freeze=False`): the embedding matrix keeps training with the rest of the
  LSTM. This is what the original script does.
- **frozen** (`freeze=True`): the embedding matrix is held fixed.

Fine-tuning on only 706 sentences moves the words that appear in the training set while
leaving every other word where the TP1 model put it. That can pull the two apart and destroy
the structure an unseen word depends on, so freezing is worth testing.

The random baseline is only run fine-tuned, since freezing random vectors would leave the
embedding layer meaningless.

In [29]:
records = []
reports = {}

for name, path in EMBEDDINGS.items():
    matrix = None
    if path is not None:
        matrix, _ = build_embedding_matrix(path, vocab_index)

    # freezing random vectors makes no sense, so the baseline is only fine-tuned
    for freeze in ([False] if matrix is None else [False, True]):
        model, history = train_model(pretrained=matrix, freeze=freeze, verbose=False)
        y_true, y_pred = predict_tags(model, testloader)

        mode = 'frozen' if freeze else 'fine-tuned'
        records.append({
            'embeddings': name,
            'mode':       mode,
            'precision':  precision_score(y_true, y_pred, zero_division=0),
            'recall':     recall_score(y_true, y_pred, zero_division=0),
            'f1':         f1_score(y_true, y_pred, zero_division=0),
            'epochs':     len(history['val_loss']),
        })
        reports[(name, mode)] = classification_report(y_true, y_pred, digits=4, zero_division=0)

        r = records[-1]
        print(f"{name:20s} {mode:11s} P {r['precision']:.4f}  R {r['recall']:.4f}  F1 {r['f1']:.4f}")

results = pd.DataFrame(records)

random               fine-tuned  P 0.3770  R 0.1581  F1 0.2227


med_w2v_cbow         fine-tuned  P 0.3408  R 0.2925  F1 0.3148


med_w2v_cbow         frozen      P 0.4522  R 0.2673  F1 0.3360


med_w2v_skipgram     fine-tuned  P 0.3378  R 0.2234  F1 0.2689


med_w2v_skipgram     frozen      P 0.4331  R 0.3002  F1 0.3546


...

press_w2v_cbow       frozen      P 0.4519  R 0.3348  F1 0.3846


press_w2v_skipgram   fine-tuned  P 0.3221  R 0.1888  F1 0.2381


press_w2v_skipgram   frozen      P 0.4518  R 0.3551  F1 0.3977


press_ft_cbow        fine-tuned  P 0.4457  R 0.3557  F1 0.3956


press_ft_cbow        frozen      P 0.4884  R 0.3798  F1 0.4273


### Results


In [30]:
# every run, best first
display(results.round(4).sort_values('f1', ascending=False).reset_index(drop=True))

# fine-tuned vs frozen, side by side
side = results.pivot(index='embeddings', columns='mode', values='f1')
side['difference'] = (side['frozen'] - side['fine-tuned']).round(4)
display(side.sort_values('fine-tuned', ascending=False))

,embeddings,mode,precision,recall,f1,epochs
0,press_ft_cbow,frozen,0.4884,0.3798,0.4273,19
1,press_w2v_skipgram,frozen,0.4518,0.3551,0.3977,27
2,press_ft_cbow,fine-tuned,0.4457,0.3557,0.3956,14
3,press_w2v_cbow,frozen,0.4519,0.3348,0.3846,17
4,press_w2v_cbow,fine-tuned,0.4141,0.3386,0.3726,17
5,med_w2v_skipgram,frozen,0.4331,0.3002,0.3546,27
6,med_ft_cbow,fine-tuned,0.3620,0.3469,0.3543,16
7,med_w2v_cbow,frozen,0.4522,0.2673,0.3360,27
8,med_w2v_cbow,fine-tuned,0.3408,0.2925,0.3148,17
9,med_ft_cbow,frozen,0.3367,0.2366,0.2779,29


mode,fine-tuned,frozen,difference
embeddings,,,
press_ft_cbow,0.395604,0.427292,0.0317
press_w2v_cbow,0.372585,0.384615,0.0120
med_ft_cbow,0.354260,0.277885,-0.0764
med_w2v_cbow,0.314826,0.335978,0.0212
med_w2v_skipgram,0.268913,0.354619,0.0857
press_w2v_skipgram,0.238062,0.397664,0.1596
random,0.222738,NaN,NaN


### Per-entity detail of the best model


In [31]:
best = results.loc[results['f1'].idxmax()]
print(f"best: {best['embeddings']} ({best['mode']}), F1 {best['f1']:.4f}\n")
print(reports[(best['embeddings'], best['mode'])])

best: press_ft_cbow (frozen), F1 0.4273

              precision    recall  f1-score   support

        ANAT     0.2000    0.0550    0.0863       109
        CHEM     0.5515    0.4213    0.4777       762
        DEVI     0.0000    0.0000    0.0000        76
        DISO     0.2280    0.3333    0.2708       249
        GEOG     0.5000    0.0714    0.1250        14
        LIVB     0.7265    0.6953    0.7105       233
        OBJC     1.0000    0.0204    0.0400        49
        PHEN     0.0000    0.0000    0.0000        29
        PHYS     0.0000    0.0000    0.0000        73
        PROC     0.5728    0.5175    0.5438       228

   micro avg     0.4884    0.3798    0.4273      1822
   macro avg     0.3779    0.2114    0.2254      1822
weighted avg     0.4691    0.3798    0.4029      1822



### Saving the scores

Written next to the notebook so the LSTM and CNN tables can be put side by side later.


In [32]:
results.to_csv(f'results_lstm_{CORPUS}.csv', index=False)
print(f'saved results_lstm_{CORPUS}.csv')

saved results_lstm_EMEA.csv
